# 🚀 DRL Trading Agent — Colab Master Pipeline

This notebook executes the **full end-to-end pipeline** for the Deep Reinforcement Learning trading agent:

1. **Stage 1 — Data Pipeline**: Fetch OHLCV → Engineer 31 features → Save `.parquet` → Push to GitHub
2. **Stage 2 — Training**: Load parquet → Train DQN (500 episodes) → Push model weights every 50 eps
3. **Stage 3 — Backtesting**: Load parquet + weights → Run OOS backtest → Push reports to GitHub

All artifacts (data, models, reports) are auto-committed back to the GitHub repository — **nothing is lost if Colab disconnects.**

---
## Cell 1 — GPU Check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU available: {gpu_name}")
    print(f"   CUDA version: {torch.version.cuda}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Training will run on CPU (slower).")
    print("   Go to Runtime → Change runtime type → Select GPU.")

---
## Cell 2 — GitHub Authentication & Clone

Enter your **GitHub Personal Access Token (PAT)** when prompted.  
Create one at: [github.com/settings/tokens](https://github.com/settings/tokens) → **Classic** → Scope: `repo`

In [ ]:
import os
from getpass import getpass

# ── Configuration ──────────────────────────────────────────────────────
GITHUB_REPO = "bhavya-goel-11/drl-agent"  # owner/repo
BRANCH = "main"
CLONE_DIR = "/content/drl-agent"

# ── Authenticate ──────────────────────────────────────────────────────
PAT = getpass("🔑 Enter your GitHub Personal Access Token: ")
CLONE_URL = f"https://{PAT}@github.com/{GITHUB_REPO}.git"

# ── Clone or Pull ─────────────────────────────────────────────────────
if os.path.exists(CLONE_DIR):
    print(f"📂 Repo already exists at {CLONE_DIR}. Pulling latest…")
    !cd {CLONE_DIR} && git pull origin {BRANCH}
else:
    print(f"📥 Cloning {GITHUB_REPO}…")
    !git clone --branch {BRANCH} {CLONE_URL} {CLONE_DIR}

# ── Set Working Directory ─────────────────────────────────────────────
os.chdir(CLONE_DIR)
print(f"\n✅ Working directory: {os.getcwd()}")

# ── Configure Git Identity ────────────────────────────────────────────
!git config user.name "Colab Pipeline"
!git config user.email "colab@pipeline.bot"
print("✅ Git identity configured.")

---
## Cell 3 — Install Dependencies

Installs Python packages from `requirements.txt` and sets up Git LFS for large file tracking.

In [ ]:
# ── Create venv & install packages ────────────────────────────────────
!python3 -m venv venv
!./venv/bin/pip install --upgrade pip -q
!./venv/bin/pip install -r requirements.txt -q

# ── Install Git LFS ───────────────────────────────────────────────────
!apt-get update -qq && apt-get install -y -qq git-lfs > /dev/null 2>&1
!git lfs install
!git lfs pull

print("\n✅ All dependencies installed.")
print("✅ Git LFS initialized.")

# Quick sanity check
!./venv/bin/python3 -c "import torch; import pandas; import yfinance; print('All imports OK')"

---
## Cell 4 — Stage 1: Data Pipeline

Fetches OHLCV data for all 47 Nifty-50 stocks via `yfinance`, engineers 31 technical features, saves as `data/processed_data.parquet`, and pushes to GitHub.

In [ ]:
!./venv/bin/python3 -m data_pipeline.loader

---
## Cell 5 — Stage 2: DRL Training

Trains the Deep Q-Network on pre-2023 data (500 episodes). Model weights are auto-saved to `models/best_model.pth` and git-pushed every 50 episodes.

In [ ]:
!./venv/bin/python3 -m drl_models.train

---
## Cell 6 — Stage 3: Out-of-Sample Backtesting

Loads the trained model and runs a full backtest on post-2023 unseen data. Reports (equity curves, alpha charts, metrics CSVs) are saved to `reports/` and pushed to GitHub.

In [ ]:
!./venv/bin/python3 -m backtesting.evaluate

---
## ✅ Pipeline Complete

All artifacts have been pushed to your GitHub repository:

| Artifact | Location |
|----------|----------|
| Processed Features | `data/processed_data.parquet` |
| Best Model Weights | `models/best_model.pth` |
| Final Model Weights | `models/final_model.pth` |
| Equity Curve | `reports/equity_curve.png` |
| Alpha Distribution | `reports/alpha_distribution.png` |
| Ticker Metrics | `reports/ticker_comparative_metrics.csv` |
| Trade Ledger | `reports/detailed_trade_ledger.csv` |